# Classic Usage of DFM Pipeline
This notebook demonstrates the classical usage of the dynamic factor model pipeline, replying organically on the `Options` class and an Excel spreadsheet containing model specifications.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from dfm_sp import Options
from dfm_sp.sp_run import run_with_options

## 1. Setup Options
The `Options` class encapsulates all runtime parameters such as iterations, cache behavior, model identifiers, and thresholds. 

Behind the scenes, it manages looking up the correct Excel Spec file and the correct Vintage historical data spreadsheet.

In [2]:
opt = Options()
opt.max_iter = 50
opt.threshold = 1e-4
opt.use_cache = False  # Set to True to skip re-running EM if cache exists

# If you leave vintage blank, it will automatically find the latest data file in the data/ folder.
# opt.vintage = '2017-01-27' 

# Print the options configuration
print(opt)

Program will be running with max_iter less than 5000! 2
DFM Runtime Options
----------------------------------------
Country         : US
Root Path       : .
Spec File       : Spec_US_example.xls
Data Folder     : data/US
Vintage         : 2017-01-27 (Requested: auto)
Sample Start    : 2000-01-01
Max Iterations  : 50
EM Threshold    : 0.0001
Use Cache       : False
Hash Key        : 1393b7a
----------------------------------------


## 2. Run Dynamic Factor Model
This `run_dfm` function abstracts the entire pipeline. It will:
- Automatically load the spec from the Excel file (`.xls`).
- Load the macro data.
- Sort, align, and apply mathematical transformations (logs, differences).
- Handle missing variables (ragged edges).
- Compile the Kalmen Filter via **Numba JIT** and execute the Expectation-Maximization loop.
- Render detailed interactive Plotly outputs and an HTML report.

In [3]:
Spec, X, Time, Z = run_with_options(opt)

from dfm_sp.sp_run import run
ResObject = run(X, Spec, opt)
Res = ResObject.result


 Table 1: Model specification 

              SeriesID                   SeriesName                 Units  \
0               PAYEMS           Payroll Employment  Thousands of Persons   
1               JTSJOL                 Job Openings             Thousands   
2             CPIAUCSL         Consumer Price Index                 Index   
3              DGORDER         Durable Goods Orders           $, Millions   
4                RSAFS                 Retail Sales           $, Millions   
5               UNRATE            Unemployment Rate                     %   
6                HOUST               Housing Starts    Thousands of Units   
7               INDPRO        Industrial Production                 Index   
8              DSPIC96              Personal Income   Chained $, Billions   
9              BOPTEXP                      Exports           $, Millions   
10             BOPTIMP                      Imports           $, Millions   
11             TTLCONS        Construction 

## 3. Review Outputs
The `Res` dictionary contains all estimation outputs, allowing you to manually slice matrices or load them into DataFrames.

In [4]:
print(f"Final EM Log-Likelihood: {Res['loglik'][-1]:.2f}")
print(f"Total Iterations Run: {len(Res['loglik']) - 1}")

print("\nAvailable matrices mapped in Results:")
print(list(Res.keys()))

Final EM Log-Likelihood: -1281.47
Total Iterations Run: 50

Available matrices mapped in Results:
['x_sm', 'X_sm', 'Z', 'C', 'R', 'A', 'Q', 'Mx', 'Wx', 'Z_0', 'V_0', 'r', 'p', 'loglik']


## 4. Visualizations and Graphing
The `dfm_sp` package provides a robust suite of plotting tools utilizing `plotly` to graph the log-likelihood convergence, factor contributions, prediction intervals, and cross-series projections.

In [5]:
from dfm_sp.sp_plots import plot_loglik, plot_common
from dfm_sp.sp_plots2 import (
    plot_factor_contribution,
    plot_factors_with_series,
    plot_prediction_intervals
)
from dfm_sp.sp_plots import plot_projection_x_over_y
from dfm_sp.sp_plots3 import plot_covariance_network, plot_em_step_deltas

# Configure to display inline plots
SHOW = True

### Log-Likelihood & Common Factors
Visualize the convergence of the EM Algorithm and map out the overall global unobserved common factor logic extracted from the Macro variables.

In [6]:
plot_loglik(ResObject, Time, show=SHOW)
plot_common(ResObject, Time, show=SHOW)

### Statistical Prediction Intervals
Plot confidence bands bounding the estimated dynamic factor over time.

In [7]:
plot_prediction_intervals(ResObject, Time, show=SHOW)

### Factor Impacts & Loading Matrices
Deconstruct how heavily each component block (Global, Real, Labor, Soft) influenced the macroeconomic unobserved factors mathematically.

In [8]:
plot_factor_contribution(ResObject, show=SHOW)

# Plot specified highly tracked series
plot_factors_with_series(ResObject, Time, opt.plot1_series, show=SHOW)

### Custom Projections against Variables
Project standard series against their Numba estimated DFM state paths to check the noise smoothing capability!

In [9]:
for items in opt.plot2_series:
    plot_projection_x_over_y(ResObject, X, Z, Time, items, show=SHOW)

### Advanced Diagnostics
Plot the Covariance error network of your specifications, and track the mathematical asymptote delta of your EM Convergence algorithm.

## 5. Automated HTML Report Generator
You can wrap an entire snapshot of the state matrices, specifications, correlations, and DFM predictions into a massive portable `.html` dashboard document seamlessly.

In [10]:
plot_covariance_network(ResObject, show=SHOW)
plot_em_step_deltas(ResObject, show=SHOW)

In [11]:
from dfm_sp import generate_html_report
generate_html_report(ResObject, Time, X, Z, opt)

HTML report saved to report-V-2017-01-27_TEST-RUN-with-max_iter50_T0.0001-1393b7a.html
